# Parasole - Heat Monitoring Dashboard
**Milan and Mexico City**

Client dashboard for the urban heat monitoring system. Connects to the Flask backend over HTTP.

Start the backend first (`python app.py`) before running.

## 1. Imports and config

In [12]:
import requests
import pandas as pd
import folium
from folium import plugins
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, FileLink, clear_output

BASE_URL = "http://localhost:5000/api/v1"
client = requests.Session()

## 2. Backend functions

GET requests to the REST API. Endpoints: /cities, /stations, /weather, /heat-stress.

In [13]:
def get_cities():
    try:
        resp = client.get(f"{BASE_URL}/cities", timeout=10)
        return resp.json() if resp.ok else []
    except requests.ConnectionError:
        print("Backend not running. Start it with: python app.py")
        return []


def get_stations(city_name):
    try:
        resp = client.get(f"{BASE_URL}/stations", params={"name": city_name}, timeout=10)
        return resp.json() if resp.ok else []
    except requests.ConnectionError:
        print("Can't connect to backend.")
        return []


def _show_backend_error(resp):
    # backend sends an explanatory message on 400/404/503 (e.g. quota exhausted)
    try:
        msg = resp.json().get("message", "Unknown error")
    except Exception:
        msg = f"HTTP {resp.status_code}"
    print(f"Backend error: {msg}")


def get_weather(station_id, start_date, end_date):
    try:
        resp = client.get(f"{BASE_URL}/weather", params={
            "station_id": station_id,
            "start_date": str(start_date),
            "end_date":   str(end_date),
        }, timeout=15)
        if resp.ok:
            df = pd.DataFrame(resp.json().get("series", []))
            if not df.empty:
                df["timestamp"] = pd.to_datetime(df["timestamp"])
            return df
        else:
            _show_backend_error(resp)
    except requests.ConnectionError:
        print("Can't connect to backend.")
    return pd.DataFrame()


def get_heat_stress(station_id, start_date, end_date, temp_th=30, hum_th=30):
    # UC4 - backend returns the critical hours
    try:
        resp = client.get(f"{BASE_URL}/heat-stress", params={
            "station_id": station_id, "start_date": str(start_date),
            "end_date": str(end_date), "temp_threshold": temp_th,
            "humidity_threshold": hum_th,
        }, timeout=15)
        if resp.ok:
            return resp.json()
        else:
            _show_backend_error(resp)
    except requests.RequestException:
        print("Can't connect to backend.")
    return None

## 3. Map

Folium map with station markers and a temperature heatmap overlay.

In [14]:
def _temp_color(t, t_min, t_max):
    # relative temperature scale: green = coolest, red = hottest
    if t_max == t_min:
        ratio = 0.5
    else:
        ratio = (t - t_min) / (t_max - t_min)

    if ratio < 0.25:
        return "#22C55E"
    elif ratio < 0.5:
        return "#EAB308"
    elif ratio < 0.75:
        return "#F97316"
    else:
        return "#EF4444"


def create_station_map(city, stations, temps=None):
    # center the map on the chosen city
    if city == "Milan":
        center = [45.46, 9.19]
    else:
        center = [19.43, -99.13]

    m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

    # build the heatmap layer (only if we have temperatures)
    if temps:
        heat = []
        for s in stations:
            if s["station_id"] in temps:
                lat = s["latitude"]
                lon = s["longitude"]
                temp = temps[s["station_id"]]
                heat.append([lat, lon, temp])

        if len(heat) > 0:
            gradient = {0.0: "#22C55E", 0.4: "#EAB308", 0.7: "#F97316", 1.0: "#EF4444"}
            plugins.HeatMap(heat, radius=40, blur=30, min_opacity=0.5,
                            gradient=gradient).add_to(m)

    # find the coldest and hottest temps, to color the markers
    if temps and len(temps) > 0:
        t_min = min(temps.values())
        t_max = max(temps.values())
    else:
        t_min = 0
        t_max = 1

    # add a marker for each station
    for s in stations:
        # get this station's temperature (if we have it)
        if temps and s["station_id"] in temps:
            t = temps[s["station_id"]]
        else:
            t = None

        # pick the marker color and popup text
        if t != None:
            color = _temp_color(t, t_min, t_max)
            extra = "<br><b>Avg Temp:</b> " + str(round(t, 1)) + "°C"
            tooltip = s["name"] + ": " + str(round(t, 1)) + "°C"
        else:
            color = "steelblue"
            extra = ""
            tooltip = s["name"]

        popup = "<b>" + s["name"] + "</b><br>ID: " + s["station_id"] + extra

        folium.CircleMarker(
            location=[s["latitude"], s["longitude"]],
            radius=11,
            color="white",
            weight=2,
            fill=True,
            fill_color=color,
            fill_opacity=0.9,
            popup=folium.Popup(popup, max_width=200),
            tooltip=tooltip,
        ).add_to(m)

    return m

## 4. Charts

Dual-axis chart (temperature + humidity), single-parameter chart, and daily precipitation.

In [15]:
def create_dual_chart(df, station_name, temp_th=35):
    # temperature and humidity together, with two y-axes
    if df.empty:
        print("No data to plot.")
        return None

    fig = go.Figure()

    # temperature line (left axis)
    fig.add_trace(go.Scatter(
        x=df["timestamp"],
        y=df["temp"],
        mode="lines",
        name="Temperature (°C)",
        line=dict(color="#EF4444", width=2),
        yaxis="y1"
    ))

    # humidity line (right axis)
    fig.add_trace(go.Scatter(
        x=df["timestamp"],
        y=df["rhum"],
        mode="lines",
        name="Humidity (%)",
        line=dict(color="#3B82F6", width=2),
        yaxis="y2"
    ))

    # dashed line for the temperature threshold
    fig.add_hline(y=temp_th, line_dash="dash", line_color="#F59E0B")

    fig.update_layout(
        title="Temperature and Humidity - " + station_name,
        xaxis_title="Time",
        yaxis=dict(title="Temperature (°C)"),
        yaxis2=dict(title="Humidity (%)", overlaying="y", side="right"),
        template="plotly_white",
        height=380
    )
    return fig


def create_trend_chart(df, station_name, param="temp", temp_th=35):
    # plot one parameter chosen in the dropdown
    if df.empty:
        print("No data to plot.")
        return None

    # pick the label and color for the chosen parameter
    if param == "temp":
        label = "Temperature (°C)"
        color = "#EF4444"
    elif param == "rhum":
        label = "Humidity (%)"
        color = "#3B82F6"
    else:
        label = "Precipitation (mm)"
        color = "#06B6D4"

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["timestamp"],
        y=df[param],
        mode="lines",
        name=label,
        line=dict(color=color, width=2)
    ))

    # only show the threshold line when looking at temperature
    if param == "temp":
        fig.add_hline(y=temp_th, line_dash="dash", line_color="#F59E0B")

    fig.update_layout(
        title=label + " - " + station_name,
        xaxis_title="Time",
        yaxis_title=label,
        template="plotly_white",
        height=380
    )
    return fig


def create_precip_chart(df, station_name):
    # add up the hourly rain into daily totals
    if df.empty:
        return None

    daily = df.groupby(df["timestamp"].dt.date)["prcp"].sum().reset_index()
    daily.columns = ["date", "precipitation"]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=daily["date"],
        y=daily["precipitation"],
        marker_color="#06B6D4"
    ))
    fig.update_layout(
        title="Daily Precipitation - " + station_name,
        xaxis_title="Date",
        yaxis_title="mm",
        template="plotly_white",
        height=280
    )
    return fig

## 5. Alerts

Color-coded banner for the city and a CSV export of the critical periods.

In [16]:
def show_alert_banner(city, stations, start_date, end_date, temp_th, hum_th):
    # add up all the critical hours from every station in the city
    total_hours = 0
    for s in stations:
        stress = get_heat_stress(s["station_id"], start_date, end_date, temp_th, hum_th)
        if stress and stress["critical_hours"] > 0:
            total_hours = total_hours + stress["critical_hours"]

    # decide the color: red if a lot of critical hours, yellow if some, green if none
    if total_hours > 10:
        color = "#dc3545"
        icon = "🔴"
        level = "CRITICAL"
    elif total_hours > 0:
        color = "#ffc107"
        icon = "🟡"
        level = "WARNING"
    else:
        color = "#28a745"
        icon = "🟢"
        level = "NORMAL"

    # build the message
    if total_hours > 0:
        msg = str(total_hours) + " critical hours in " + city
    else:
        msg = "No critical events in " + city

    # show the colored banner
    banner = f"""
    <div style="padding:12px 18px; background:{color}15; border:2px solid {color}40;
                border-radius:8px; margin:6px 0; font-family:sans-serif;">
        <span style="font-size:20px;">{icon}</span>
        <b style="color:{color};"> {level}</b> - {msg}
        <br><span style="font-size:11px; color:#888;">Thresholds: Temp > {temp_th}°C AND Humidity > {hum_th}%</span>
    </div>
    """
    display(HTML(banner))

def export_alert_report(station_id, station_name, city, start_date, end_date, temp_th, hum_th):
    # get the critical periods from the backend
    stress = get_heat_stress(station_id, start_date, end_date, temp_th, hum_th)

    # if there is no data or no critical hours, stop here
    if stress == None:
        print("No data to export")
        return
    if stress["critical_hours"] == 0:
        print("No critical events to export")
        return

    # put the critical periods into a table
    periods = stress["critical_periods"]
    df = pd.DataFrame(periods)

    # build the file name (replace spaces with underscores so it's a valid name)
    st_name = station_name.replace(" ", "_")
    st_name = st_name.replace("/", "_")
    st_name = st_name.replace("\\", "_")
    filename = "alert_" + st_name + ".csv"

    # save it and show a download link
    df.to_csv(filename, index=False)
    print("Report exported: " + filename)
    return FileLink(filename)

## 6. Summary indicators

Descriptive statistics shown as cards.

In [17]:
def show_summary(df, critical_hours=0, n_stations=0):
    if df.empty:
        print("No data for summary.")
        return

    # compute the statistics
    avg_temp = round(df["temp"].mean(), 1)
    max_temp = round(df["temp"].max(), 1)
    min_temp = round(df["temp"].min(), 1)
    avg_hum = round(df["rhum"].mean(), 0)
    total_prcp = round(df["prcp"].sum(), 1)

    # build the list of cards to show
    stats = []
    stats.append(("Avg Temp", str(avg_temp) + "°C", "#EF4444"))
    stats.append(("Max Temp", str(max_temp) + "°C", "#EF4444"))
    stats.append(("Min Temp", str(min_temp) + "°C", "#06B6D4"))
    stats.append(("Avg Humidity", str(avg_hum) + "%", "#3B82F6"))
    stats.append(("Total Precip", str(total_prcp) + " mm", "#06B6D4"))
    stats.append(("Critical Hrs", str(critical_hours) + "h", "#F59E0B"))
    stats.append(("Active Stations", str(n_stations), "#28a745"))

    # turn each stat into an HTML card
    cards = ""
    for stat in stats:
        label = stat[0]
        value = stat[1]
        color = stat[2]
        cards = cards + f"""
        <div style="padding:8px 12px; border-radius:6px; background:{color}10;
                    border:1px solid {color}20; text-align:center; min-width:100px;">
            <div style="font-size:10px; color:#888; text-transform:uppercase;">{label}</div>
            <div style="font-size:18px; font-weight:700; color:{color};">{value}</div>
        </div>"""

    display(HTML("<div style='display:flex; gap:8px; flex-wrap:wrap; margin:8px 0;'>" + cards + "</div>")) 

## 7. Build the dashboard

Widgets, event handlers, and layout.

In [18]:
# widgets
city_dd  = widgets.Dropdown(options=["Milan", "Mexico City"], value="Milan", description="City:")
search   = widgets.Text(placeholder="Search station...", description="🔍")
station_dd = widgets.Dropdown(options=[], description="Station:")
param_dd = widgets.Dropdown(options=[("Temperature", "temp"), ("Humidity", "rhum"),
                                     ("Precipitation", "prcp")], value="temp", description="Param:")
start_dt = widgets.DatePicker(description="From:", value=pd.Timestamp("2026-06-01").date())
end_dt   = widgets.DatePicker(description="To:",   value=pd.Timestamp("2026-06-07").date())
temp_sl  = widgets.IntSlider(value=30, min=20, max=50, description="Temp (°C):", readout_format="d")
hum_sl   = widgets.IntSlider(value=30, min=15, max=100, step=5, description="Hum (%):", readout_format="d")
load_btn = widgets.Button(description="Load Data", button_style="info", icon="refresh")
exp_btn  = widgets.Button(description="Export Alert Report", button_style="warning",
                          layout=widgets.Layout(width="100%"))

# one output box per panel
banner_out  = widgets.Output()
map_out     = widgets.Output()
dual_out    = widgets.Output()
chart_out   = widgets.Output()
precip_out  = widgets.Output()
alerts_out  = widgets.Output()
summary_out = widgets.Output()
export_out  = widgets.Output()

# shared state used by the event handlers
current_df = pd.DataFrame()
current_station = None
current_stations = []

# remembers the city and dates used for the last city map,
# so we only redraw it when one of them changes
last_view = {"city": None, "start": None, "end": None}

### Event handlers

In [19]:
def find_station(name):
    # look through the stations and return the one with this name
    for s in current_stations:
        if s["name"] == name:
            return s
    return None


def on_city(change):
    # when the city changes, load its stations into the dropdown
    global current_stations
    new_city = change["new"]
    current_stations = get_stations(new_city)

    # build the list of station names for the dropdown
    names = []
    for s in current_stations:
        names.append(s["name"])
    station_dd.options = names

    search.value = ""

    # show the city-wide map and banner (no station needed yet)
    show_city_view()


def on_search(change):
    # filter the dropdown as the user types
    text = change["new"].lower()

    # keep the names that contain what the user typed
    matches = []
    for s in current_stations:
        if text in s["name"].lower():
            matches.append(s["name"])

    # if nothing matches, show all the stations
    if len(matches) == 0:
        names = []
        for s in current_stations:
            names.append(s["name"])
        station_dd.options = names
    else:
        station_dd.options = matches


def on_export(_):
    # save the report for the selected station and show the result
    with export_out:
        clear_output(wait=True)
        if current_station != None:
            link = export_alert_report(current_station["station_id"], current_station["name"],
                                       city_dd.value, start_dt.value, end_dt.value,
                                       temp_sl.value, hum_sl.value)
            if link != None:
                display(link)
        else:
            print("Select a station first.")


def on_threshold_change(change):
    # runs automatically when you move the temp or humidity slider.
    # only updates the selected station's chart and critical hours,
    # not the whole city map/banner (those stay on the Load Data button).
    if current_station == None:
        return
    if current_df.empty:
        return

    # ask the backend for the new critical hours with the new thresholds
    stress = get_heat_stress(current_station["station_id"], start_dt.value, end_dt.value,
                             temp_sl.value, hum_sl.value)
    if stress == None:
        crit = 0
    else:
        crit = stress["critical_hours"]

    # redraw the temperature + humidity chart with the new threshold line
    with dual_out:
        clear_output(wait=True)
        fig_dual = create_dual_chart(current_df, current_station["name"], temp_sl.value)
        if fig_dual != None:
            fig_dual.show()

    # redraw the single-parameter chart
    with chart_out:
        clear_output(wait=True)
        fig = create_trend_chart(current_df, current_station["name"], param_dd.value, temp_sl.value)
        if fig != None:
            fig.show()

    # update the alerts panel with the new critical hours
    with alerts_out:
        clear_output(wait=True)
        if crit > 0:
            print(str(crit) + " critical hours for " + current_station["name"])
            periods = stress["critical_periods"]
            display(pd.DataFrame(periods))
        else:
            print("No critical events for " + current_station["name"])


### on_load

Runs on Load Data: fetches the data and refreshes every panel.

In [20]:
def update_city_map_and_banner():
    # draws the city-wide map (heatmap) and the alert banner.
    # used both when the city changes and when Load Data is pressed.

    # get the average temperature of each station for the heatmap
    temps = {}
    for s in current_stations:
        d = get_weather(s["station_id"], start_dt.value, end_dt.value)
        if not d.empty:
            temps[s["station_id"]] = d["temp"].mean()

    # update the alert banner (whole city)
    with banner_out:
        clear_output(wait=True)
        show_alert_banner(city_dd.value, current_stations,
                          start_dt.value, end_dt.value, temp_sl.value, hum_sl.value)

    # update the map (whole city)
    with map_out:
        clear_output(wait=True)
        m = create_station_map(city_dd.value, current_stations, temps)
        display(m)

    # remember what this map was drawn with (see last_view)
    last_view["city"] = city_dd.value
    last_view["start"] = start_dt.value
    last_view["end"] = end_dt.value


def show_city_view():
    # called when the city changes
    update_city_map_and_banner()

    # show how many stations the city has in the summary
    with summary_out:
        clear_output(wait=True)
        print("City: " + city_dd.value)
        print("Stations: " + str(len(current_stations)))
        print("Select a station to see its charts.")


def on_load(_):
    # shows the charts and alerts for the SELECTED station.
    global current_df, current_station

    # find which station the user selected
    current_station = find_station(station_dd.value)
    if current_station == None:
        return

    # get the weather data for that station
    current_df = get_weather(current_station["station_id"], start_dt.value, end_dt.value)

    # ask the backend for the critical hours (UC4)
    stress = get_heat_stress(current_station["station_id"], start_dt.value, end_dt.value,
                             temp_sl.value, hum_sl.value)
    if stress == None:
        crit = 0
    else:
        crit = stress["critical_hours"]

    # update the temperature + humidity chart
    with dual_out:
        clear_output(wait=True)
        fig_dual = create_dual_chart(current_df, current_station["name"], temp_sl.value)
        if fig_dual != None:
            fig_dual.show()

    # update the single-parameter chart
    with chart_out:
        clear_output(wait=True)
        fig = create_trend_chart(current_df, current_station["name"], param_dd.value, temp_sl.value)
        if fig != None:
            fig.show()

    # update the precipitation chart (skip if precipitation is already shown above)
    with precip_out:
        clear_output(wait=True)
        if param_dd.value != "prcp":
            fig2 = create_precip_chart(current_df, current_station["name"])
            if fig2 != None:
                fig2.show()

    # update the alerts panel
    with alerts_out:
        clear_output(wait=True)
        if crit > 0:
            print(str(crit) + " critical hours for " + current_station["name"])
            periods = stress["critical_periods"]
            display(pd.DataFrame(periods))
        else:
            print("No critical events for " + current_station["name"])

    # the city map depends on (city, dates): only redraw it when
    # one of them changed, not on every Load Data click
    if (last_view["city"] != city_dd.value
            or last_view["start"] != start_dt.value
            or last_view["end"] != end_dt.value):
        update_city_map_and_banner()

    # update the summary cards for this station
    with summary_out:
        clear_output(wait=True)
        n_stations = len(current_stations)
        show_summary(current_df, crit, n_stations=n_stations)


### Connect widgets and display

In [21]:
city_dd.observe(on_city, names="value")
search.observe(on_search, names="value")
load_btn.on_click(on_load)
exp_btn.on_click(on_export)
temp_sl.observe(on_threshold_change, names="value")
hum_sl.observe(on_threshold_change, names="value")

display(HTML("""
<div style='background:linear-gradient(135deg,#0F172A,#1B3A5C);
            padding:14px 22px; border-radius:8px; margin-bottom:10px;'>
    <span style='font-size:16px; font-weight:700; color:white;'>
        Parasole - Heat Monitoring Dashboard</span>
    <div style='font-size:10px; color:#94A3B8; margin-top:3px;'>
        SE4GEO 2026 · Milan and Mexico City</div>
</div>
"""))

display(widgets.HBox([city_dd, search, station_dd]))
display(widgets.HBox([param_dd, start_dt, end_dt]))
display(widgets.HBox([temp_sl, hum_sl, load_btn]))
display(banner_out)
display(HTML("<h3>🗺️ Station Map</h3>"))
display(map_out)
display(HTML("<h3>Temperature and Humidity</h3>"))
display(dual_out)
display(HTML("<h3>Parameter Detail</h3>"))
display(chart_out)
display(precip_out)
display(HTML("<h3>⚠️ Heat Stress Alerts</h3>"))
display(alerts_out)
display(exp_btn)
display(export_out)
display(HTML("<h3>Summary Indicators</h3>"))
display(summary_out)

on_city({"new": "Milan"})

Output()

Output()

Output()

Output()

Output()

Output()

Button(button_style='warning', description='Export Alert Report', layout=Layout(width='100%'), style=ButtonSty…

Output()

Output()

## 8. Backend check (optional)

Quick check that the backend is reachable and returning data.

In [22]:
'''# get the list of cities
cities = get_cities()
print("Cities:")
for c in cities:
    print(" -", c["name"])

# get the stations of Milan
stations = get_stations("Milan")
print("Milan has", len(stations), "stations")

# get the weather of one station and show the first rows
df = get_weather("16080", "2026-06-01", "2026-06-07")
df.head() '''

'# get the list of cities\ncities = get_cities()\nprint("Cities:")\nfor c in cities:\n    print(" -", c["name"])\n\n# get the stations of Milan\nstations = get_stations("Milan")\nprint("Milan has", len(stations), "stations")\n\n# get the weather of one station and show the first rows\ndf = get_weather("16080", "2026-06-01", "2026-06-07")\ndf.head() '